# CatBoost — Predicting Smartphone Addiction (Playground Series S6E8)

**Target:** `addicted_label` (binary)
**Metric:** AUC-ROC

This notebook trains a CatBoost classifier with native categorical handling for `gender`, `stress_level`, and `academic_work_impact`, using stratified 10-fold cross-validation.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# matplotlib backend for headless batch execution on Snellius
import matplotlib
matplotlib.use("Agg")

## 2. Load data

Paths point to the local data directory on Snellius. Adjust `DATA_DIR` if your files live elsewhere.

In [2]:
import os

DATA_DIR = os.environ.get("DATA_DIR", os.path.join(os.getcwd(), "data"))

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(train.shape, test.shape)
train.head()

(691369, 14) (296302, 13)


,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


## 3. Define features and target

In [3]:
TARGET = "addicted_label"
ID_COL = "id" if "id" in train.columns else train.columns[0]

CAT_FEATURES = ["gender", "stress_level", "academic_work_impact"]

# Everything else (minus id/target) is treated as numeric
NUM_FEATURES = [
    c for c in train.columns
    if c not in CAT_FEATURES + [TARGET, ID_COL]
]

FEATURES = NUM_FEATURES + CAT_FEATURES
print("Numeric features:", NUM_FEATURES)
print("Categorical features:", CAT_FEATURES)

Numeric features: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time']
Categorical features: ['gender', 'stress_level', 'academic_work_impact']


## 4. Quick EDA

Class balance and a peek at the categorical/numeric distributions.

In [4]:
print(train[TARGET].value_counts(normalize=True))
train[TARGET].value_counts().plot(kind="bar", title="Class balance: addicted_label")
plt.savefig("class_balance.png", bbox_inches="tight")
plt.close()

addicted_label
1    0.709424
0    0.290576
Name: proportion, dtype: float64


In [5]:
train[NUM_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
age,662440.0,26.615408,5.153162,18.00,22.00,27.00,31.00,35.00
daily_screen_time_hours,595515.0,7.640865,2.721446,0.50,5.48,7.77,9.84,15.00
social_media_hours,557374.0,2.471038,1.316137,0.00,1.45,2.31,3.37,8.00
gaming_hours,564548.0,1.459265,0.934552,0.00,0.70,1.33,2.09,4.00
work_study_hours,639851.0,2.366971,1.258797,0.00,1.36,2.20,3.20,6.00
sleep_hours,646889.0,6.804334,1.234512,4.50,5.78,6.80,7.87,9.00
notifications_per_day,623785.0,145.894900,65.917556,20.00,93.00,150.00,204.00,250.00
app_opens_per_day,610659.0,102.636781,48.093970,15.00,64.00,104.00,145.00,180.00
weekend_screen_time,579306.0,9.479866,2.856006,0.51,7.28,9.58,11.75,17.56


## 5. Prep for CatBoost

CatBoost needs categorical columns as strings (not floats/NaN-as-float) and takes their column indices via `cat_features`.

In [6]:
for col in CAT_FEATURES:
    train[col] = train[col].fillna("missing").astype(str)
    test[col] = test[col].fillna("missing").astype(str)

X = train[FEATURES]
y = train[TARGET]
X_test = test[FEATURES]

cat_feature_idx = [X.columns.get_loc(c) for c in CAT_FEATURES]
cat_feature_idx

[9, 10, 11]

## 6. Stratified 10-fold CV training

Stratification keeps the class ratio consistent per fold. `eval_metric="AUC"` with early stopping optimizes directly for the competition metric.

In [7]:
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []
models = []

params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=42,
    early_stopping_rounds=200,
    verbose=200,
    task_type="GPU",
    devices="0",
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_pool = Pool(X_train, y_train, cat_features=cat_feature_idx)
    val_pool = Pool(X_val, y_val, cat_features=cat_feature_idx)
    test_pool = Pool(X_test, cat_features=cat_feature_idx)

    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    models.append(model)

    val_pred = model.predict_proba(val_pool)[:, 1]
    oof_preds[val_idx] = val_pred

    fold_auc = roc_auc_score(y_val, val_pred)
    fold_scores.append(fold_auc)
    print(f"Fold {fold}: AUC = {fold_auc:.5f}")

    test_preds += model.predict_proba(test_pool)[:, 1] / N_SPLITS

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9058316	best: 0.9058316 (0)	total: 34.8ms	remaining: 1m 44s


200:	test: 0.9364746	best: 0.9364746 (200)	total: 3.18s	remaining: 44.3s


400:	test: 0.9425056	best: 0.9425056 (400)	total: 6.3s	remaining: 40.9s


600:	test: 0.9461851	best: 0.9461851 (600)	total: 9.47s	remaining: 37.8s


...

2200:	test: 0.9557105	best: 0.9557105 (2200)	total: 34.6s	remaining: 12.5s


2400:	test: 0.9560466	best: 0.9560466 (2400)	total: 37.7s	remaining: 9.4s


2600:	test: 0.9563299	best: 0.9563299 (2600)	total: 40.8s	remaining: 6.27s


2800:	test: 0.9565932	best: 0.9565932 (2800)	total: 44s	remaining: 3.13s


2999:	test: 0.9568199	best: 0.9568199 (2999)	total: 47.1s	remaining: 0us
bestTest = 0.9568198919
bestIteration = 2999
Fold 9: AUC = 0.95682


## 7. Overall CV score

In [8]:
overall_auc = roc_auc_score(y, oof_preds)
print(f"Mean fold AUC: {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}")
print(f"Overall OOF AUC: {overall_auc:.5f}")

Mean fold AUC: 0.95810 +/- 0.00075
Overall OOF AUC: 0.95810


## 8. Feature importance

Sanity check on which features CatBoost is relying on most.

In [9]:
importances = models[-1].get_feature_importance(train_pool)
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": importances,
}).sort_values("importance", ascending=False)

importance_df.plot(x="feature", y="importance", kind="barh", figsize=(8, 5), legend=False)
plt.gca().invert_yaxis()
plt.title("CatBoost feature importance (last fold)")
plt.tight_layout()
plt.savefig("feature_importance.png", bbox_inches="tight")
plt.close()

importance_df

,feature,importance
1,daily_screen_time_hours,26.423650
8,weekend_screen_time,24.213976
2,social_media_hours,18.279943
6,notifications_per_day,11.815003
7,app_opens_per_day,11.144608
4,work_study_hours,3.394998
3,gaming_hours,2.525029
0,age,1.081682
5,sleep_hours,1.024300
9,gender,0.064254


## 9. Build submission

In [10]:
submission = test[[ID_COL]].copy()
submission[TARGET] = test_preds
submission.to_csv("submission.csv", index=False)
submission.head()

,id,addicted_label
0,691369,0.999294
1,691370,0.944154
2,691371,0.970362
3,691372,0.973873
4,691373,0.996992
